In [102]:
import tqdm 
import sys
import pandas as pd
sys.path.append('../kaggle_prediction_library/') 
from web_scraping.torvik_player_scraping_functions import get_html_from_torvik_players, get_data_from_html

### Scrape the Data

In [103]:
all_first_round_dates = [
    "20080321", "20080320", 
    "20090320", "20090319", 
    "20100319", "20100318",
    "20110318", "20110317", 
    "20120316", "20120315", 
    "20130322", "20130321",
    "20140321", "20140320", 
    "20150320", "20150319", 
    "20160318", "20160317",
    "20170317", "20170316", 
    "20180316", "20180315", 
    "20190322", "20190321",
    "20200320", "20200319",    
    "20210319", "20210320", 
    "20220318", "20220317",
    "20230317", "20230316", 
    "20240322", "20240321"
]

# note this is the day before the first four, for safety
# day_before_tournament_start_dict = {
#     '2008': '20080317', '2009': '20090316', '2010': '20100315',
#     '2011': '20110314', '2012': '20120312', '2013': '20130317',
#     '2014': '20140317', '2015': '20150316', '2016': '20160314',
#     '2017': '20170313', '2018': '20180312', '2019': '20190318',
#     '2020': '20200316', '2021': '20210316', '2022': '20220314',
#     '2023': '20230313', '2024': '20240318'
# }

day_before_tournament_start_dict = {
   '2025': '20250324',
}

day_before_tournament_start_df = pd.DataFrame(list(day_before_tournament_start_dict.items()), columns=['year', 'day_before_tourney_start'])
day_before_tournament_start_df['season_start_date'] = (day_before_tournament_start_df['year'].astype(int) - 1).astype(str) + '1101'


In [104]:
all_dfs = []

for index, row in tqdm.tqdm(day_before_tournament_start_df.iterrows()):
    
    start = row["season_start_date"]
    end = row["day_before_tourney_start"]
    year = row["year"]

    html = get_html_from_torvik_players(year, start, end, 40)
    tmp_df = get_data_from_html(html)
    tmp_df["Season"] = year

    all_dfs.append(tmp_df)


0it [00:00, ?it/s]

Loading URL: https://barttorvik.com/playerstat.php?link=y&sIndex=53&minmin=5&year=2025&start=20241101&end=20250324


The chromedriver version (133.0.6943.53) detected in PATH at /opt/homebrew/bin/chromedriver might not be compatible with the detected chrome version (134.0.6998.119); currently, chromedriver 134.0.6998.165 is recommended for chrome 134.*, so it is advised to delete the driver in PATH and retry


Clicked on 'Games' column successfully.
'Show 100 more' clicked (1/40)
'Show 100 more' clicked (2/40)
'Show 100 more' clicked (3/40)
'Show 100 more' clicked (4/40)
'Show 100 more' clicked (5/40)
'Show 100 more' clicked (6/40)
'Show 100 more' clicked (7/40)
'Show 100 more' clicked (8/40)
'Show 100 more' clicked (9/40)
'Show 100 more' clicked (10/40)
'Show 100 more' clicked (11/40)
'Show 100 more' clicked (12/40)
'Show 100 more' clicked (13/40)
'Show 100 more' clicked (14/40)
'Show 100 more' clicked (15/40)
'Show 100 more' clicked (16/40)
'Show 100 more' clicked (17/40)
'Show 100 more' clicked (18/40)
'Show 100 more' clicked (19/40)
'Show 100 more' clicked (20/40)
'Show 100 more' clicked (21/40)
'Show 100 more' clicked (22/40)
'Show 100 more' clicked (23/40)
'Show 100 more' clicked (24/40)
'Show 100 more' clicked (25/40)
'Show 100 more' clicked (26/40)
'Show 100 more' clicked (27/40)
'Show 100 more' clicked (28/40)
'Show 100 more' clicked (29/40)
'Show 100 more' clicked (30/40)
'Show 100

1it [09:02, 542.28s/it]


In [132]:
final_df = pd.concat(all_dfs, axis=0)
final_df = final_df[final_df["Min%"].notnull()]


### Map to the Kaggle Ids 

In [133]:
tourney_teams = []

In [134]:
mapping = pd.read_csv("../data/sky_data/mappings/kaggle_torvik_mapping.csv")

In [135]:
teams = pd.read_csv("../data/MTeams.csv")

In [136]:
final_df["Final_Torvik_Team"] = final_df["Team"]
final_df = final_df.merge(mapping, how="inner", on=["Final_Torvik_Team"])

In [138]:
#final_df = final_df.rename(columns = {"Kaggle_Team_x": "Kaggle_Team"})

In [139]:
final_df["TeamName"] = final_df["Kaggle_Team"]

In [140]:
final_df = final_df.merge(teams[["TeamID", "TeamName"]], how="left", on=["TeamName"])

In [141]:
curr = pd.read_csv("../data/sky_data/torvik_player_data_2008_2024.csv")

In [142]:
to_write = pd.concat([curr[final_df.columns], final_df], axis =0)

In [143]:
final_df


,Rk,Class,Height,Player,Team,Conf,Games,Min%,PRPG!,BPM,...,AST,TO,BLK,STL,FTR,Season,Final_Torvik_Team,Kaggle_Team,TeamName,TeamID
0,1,Fr,6-9,Cooper Flagg,Duke,ACC,34,71.3,5.8,14.9,...,26.9,13.8,4.6,2.9,43.3,2025,Duke,Duke,Duke,1181
1,2,Sr,6-10,Johni Broome,Auburn,SEC,33,70.8,5.8,12.9,...,20.0,9.1,7.9,1.6,39.3,2025,Auburn,Auburn,Auburn,1120
2,3,Fr,6-3,Makhai Valentine,Missouri St.,MVC,11,5.8,0.8,12.0,...,3.8,13.6,1.8,3.5,34.5,2025,Missouri St.,Missouri St,Missouri St,1283
3,4,Sr,6-9,Yaxel Lendeborg,UAB,Amer,35,83.8,5.2,11.6,...,23.1,13.9,5.3,2.9,45.5,2025,UAB,UAB,UAB,1412
4,5,Jr,6-3,Emanuel Sharp,Houston,B12,33,60.2,4.5,11.2,...,6.0,9.9,0.0,3.4,38.1,2025,Houston,Houston,Houston,1222
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3472,4009,Jr,6-10,Darrion Salery,Mississippi Valley S,SWAC,15,12.2,-1.4,-16.1,...,1.6,26.4,7.1,0.0,21.4,2025,Mississippi Valley S,MS Valley St,MS Valley St,1290
3473,4010,So,6-10,Aleks Szymczyk,Charlotte,Amer,19,6.6,-0.4,-16.2,...,4.8,25.5,0.0,0.0,23.1,2025,Charlotte,Charlotte,Charlotte,1150
3474,4011,Fr,6-1,Trey O'Neil,Elon,CAA,10,5.1,-1.3,-16.4,...,13.5,44.4,0.0,1.9,20.0,2025,Elon,Elon,Elon,1189
3475,4012,So,6-3,Aa'reyon Munir-Jones,Coppin St.,MEAC,8,5.2,-1.0,-18.6,...,3.4,37.1,0.0,1.0,54.5,2025,Coppin St.,Coppin St,Coppin St,1164


In [144]:
to_write.to_csv("../data/sky_data/torvik_player_data_2008_2025_s16.csv", index=False)